# CSE-CIC-IDS2018 Preprocessing Notebook

This notebook is used for Stage 1 data exploration and preprocessing experiments.

The final reusable preprocessing logic should be kept in:

`stage-1/scripts/sample_cse_cic_ids2018.py`

Large raw CSV files should not be committed to GitHub.

## Dataset source note

Primary Colab source:

`kagglehub.dataset_download("solarmainframe/ids-intrusion-csv")`

Local fallback source:

`archive.zip`

The local `archive.zip` was inspected before this notebook was updated. It contains 10 CSV files named by date, such as `02-14-2018.csv`, `02-20-2018.csv`, and `03-02-2018.csv`.

Most files contain 80 columns starting with `Dst Port`, `Protocol`, and `Timestamp`, ending with `Label`. The `02-20-2018.csv` file contains 84 columns and additionally includes `Flow ID`, `Src IP`, `Src Port`, and `Dst IP`. Because of this mixed schema, the notebook uses safe column access and fallback values when source or destination IP fields are missing.

## 1. Setup

In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd

if importlib.util.find_spec('kagglehub') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'kagglehub'])

import kagglehub

KAGGLE_DATASET = 'solarmainframe/ids-intrusion-csv'
DATA_SOURCE = 'kagglehub'  # Use 'archive' to read a local archive.zip fallback.
ARCHIVE_PATH = Path('/content/archive.zip')
OUTPUT_PATH = Path('/content/sample-alerts.json')
# After reviewing the Colab output, copy the exported JSON into:
# stage-1/data/processed/sample-alerts.json
# Do not automatically overwrite the repository file from this notebook.
DATASET_DIR = None

BENIGN_SAMPLE_SIZE = 500
ATTACK_SAMPLE_SIZE = 500
RANDOM_STATE = 42

## 2. Load Dataset

In [ ]:
# Download through KaggleHub in Colab, or inspect archive.zip as a local fallback.
if DATA_SOURCE == 'kagglehub':
    dataset_path = kagglehub.dataset_download(KAGGLE_DATASET)
    DATASET_DIR = Path(dataset_path)
    csv_files = sorted(str(path.relative_to(DATASET_DIR)) for path in DATASET_DIR.rglob('*.csv'))
else:
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        csv_files = [name for name in archive.namelist() if name.lower().endswith('.csv')]

print('Dataset path:', DATASET_DIR if DATA_SOURCE == 'kagglehub' else ARCHIVE_PATH)
csv_files

In [ ]:
# Load one or more files for Stage 1 exploration.
# Start small. 02-20-2018.csv is very large and has extra IP columns.
SELECTED_FILES = [name for name in csv_files if name.endswith('02-16-2018.csv')]
if not SELECTED_FILES:
    SELECTED_FILES = csv_files[:1]

def read_csv_member(member_name, nrows=None):
    if DATA_SOURCE == 'kagglehub':
        return pd.read_csv(DATASET_DIR / member_name, nrows=nrows)

    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        with archive.open(member_name) as file:
            return pd.read_csv(file, nrows=nrows)

# Use nrows while exploring to avoid loading very large files into memory.
frames = [read_csv_member(name, nrows=100_000) for name in SELECTED_FILES]
df = pd.concat(frames, ignore_index=True)
CSV_FILES_USED = SELECTED_FILES
ROWS_LOADED = len(df)
df.head()

## 3. Inspect Columns

In [ ]:
print('Rows, columns:', df.shape)
print(df.columns.tolist())
df.info()

In [ ]:
# Inspect column differences across CSV files.
schema_summary = []
for name in csv_files:
    preview = read_csv_member(name, nrows=0)
    columns = preview.columns.tolist()
    schema_summary.append({
        'file': name,
        'column_count': len(columns),
        'has_src_ip': 'Src IP' in columns,
        'has_dst_ip': 'Dst IP' in columns,
        'has_label': 'Label' in columns,
        'first_columns': columns[:8],
    })

pd.DataFrame(schema_summary)

## 4. Clean Data

In [ ]:
def clean_column_names(dataframe):
    cleaned = dataframe.copy()
    cleaned.columns = cleaned.columns.str.strip()
    return cleaned

def clean_invalid_values(dataframe):
    cleaned = dataframe.copy()
    cleaned = cleaned.replace([np.inf, -np.inf], np.nan)
    cleaned = cleaned.dropna(subset=['Label'])
    return cleaned

df = clean_column_names(df)
df = clean_invalid_values(df)
ROWS_AFTER_CLEANING = len(df)
ROWS_REMOVED_DURING_CLEANING = ROWS_LOADED - ROWS_AFTER_CLEANING
df.head()

## 5. Check Label Distribution

In [ ]:
label_counts = df['Label'].value_counts(dropna=False)
LABEL_DISTRIBUTION_BEFORE_SAMPLING = label_counts.to_dict()
label_counts

## 6. Sample Benign and Attack Rows

In [ ]:
def sample_benign_and_attack_rows(dataframe, benign_n=BENIGN_SAMPLE_SIZE, attack_n=ATTACK_SAMPLE_SIZE):
    """Sample benign rows plus attack rows stratified by raw Label."""
    labels = dataframe['Label'].astype(str)
    benign_rows = dataframe[labels.str.lower() == 'benign']
    attack_rows = dataframe[labels.str.lower() != 'benign']

    benign_sample = benign_rows.sample(
        n=min(benign_n, len(benign_rows)), random_state=RANDOM_STATE
    )

    attack_groups = []
    sampled_attack_indices = []
    attack_labels = sorted(attack_rows['Label'].astype(str).unique())
    if attack_labels:
        per_label_target = max(1, attack_n // len(attack_labels))
        for label in attack_labels:
            label_rows = attack_rows[attack_rows['Label'].astype(str) == label]
            label_sample = label_rows.sample(
                n=min(per_label_target, len(label_rows)), random_state=RANDOM_STATE
            )
            attack_groups.append(label_sample)
            sampled_attack_indices.extend(label_sample.index.tolist())

    attack_sample = pd.concat(attack_groups) if attack_groups else attack_rows.head(0)

    if len(attack_sample) < min(attack_n, len(attack_rows)):
        remaining = attack_rows.drop(index=sampled_attack_indices, errors='ignore')
        extra_needed = min(attack_n, len(attack_rows)) - len(attack_sample)
        if extra_needed > 0 and len(remaining) > 0:
            extra_sample = remaining.sample(n=min(extra_needed, len(remaining)), random_state=RANDOM_STATE)
            attack_sample = pd.concat([attack_sample, extra_sample])

    return pd.concat([benign_sample, attack_sample], ignore_index=True)

sample_df = sample_benign_and_attack_rows(df)
LABEL_DISTRIBUTION_AFTER_SAMPLING = sample_df['Label'].value_counts(dropna=False).to_dict()
BENIGN_SAMPLE_COUNT = int((sample_df['Label'].astype(str).str.lower() == 'benign').sum())
MALICIOUS_SAMPLE_COUNT = int((sample_df['Label'].astype(str).str.lower() != 'benign').sum())
sample_df['Label'].value_counts(dropna=False)

## 7. Convert Rows to Dashboard Alerts

In [ ]:
PROTOCOL_MAP = {
    0: 'HOPOPT',
    1: 'ICMP',
    6: 'TCP',
    17: 'UDP',
}

def map_label_to_attack_type(label: str) -> str:
    normalized = str(label).strip().lower()
    if normalized == 'benign':
        return 'Benign'
    if normalized in {'dos hulk', 'dos goldeneye', 'dos slowloris', 'dos slowhttptest'}:
        return 'DoS'
    if 'ddos' in normalized:
        return 'DDoS'
    if normalized in {'ftp-bruteforce', 'ssh-bruteforce', 'ftp-patator', 'ssh-patator'}:
        return 'Brute Force'
    if normalized == 'bot':
        return 'Botnet'
    if 'web attack' in normalized or normalized in {'sql injection', 'xss', 'brute force -web'}:
        return 'Web Attack'
    if 'infilteration' in normalized or 'infiltration' in normalized:
        return 'Infiltration'
    return 'Unknown Attack'

def map_attack_type_to_severity(attack_type: str) -> str:
    return {
        'Benign': 'Low',
        'Brute Force': 'Medium',
        'Web Attack': 'High',
        'Botnet': 'High',
        'DoS': 'High',
        'DDoS': 'High',
        'Infiltration': 'Critical',
        'Unknown Attack': 'Medium',
    }.get(attack_type, 'Medium')

def map_attack_type_to_risk_score(attack_type: str) -> int:
    return {
        'Benign': 25,
        'Brute Force': 68,
        'Web Attack': 76,
        'Botnet': 80,
        'DoS': 84,
        'DDoS': 88,
        'Infiltration': 92,
        'Unknown Attack': 64,
    }.get(attack_type, 64)

def safe_value(row, field, default='unknown'):
    if field not in row.index:
        return default
    value = row[field]
    if pd.isna(value):
        return default
    return value

def protocol_name(value):
    try:
        return PROTOCOL_MAP.get(int(value), str(value))
    except (TypeError, ValueError):
        return str(value)

def row_to_alert(row, index):
    label = str(safe_value(row, 'Label', 'Benign'))
    attack_type = map_label_to_attack_type(label)
    is_benign = attack_type == 'Benign'
    risk_score = map_attack_type_to_risk_score(attack_type)
    source_ip = str(safe_value(row, 'Src IP', 'unknown-source'))
    destination_ip = str(safe_value(row, 'Dst IP', 'unknown-destination'))
    port = int(float(safe_value(row, 'Dst Port', 0)))
    protocol = protocol_name(safe_value(row, 'Protocol', 'TCP'))

    return {
        'id': f'AL-{index:04d}',
        'timestamp': str(safe_value(row, 'Timestamp', 'unknown-time')),
        'sourceIp': source_ip,
        'destinationIp': destination_ip,
        'protocol': protocol,
        'port': port,
        'attackType': attack_type,
        'severity': map_attack_type_to_severity(attack_type),
        'confidence': risk_score,
        'baseRiskScore': risk_score,
        'currentRiskScore': risk_score,
        'status': 'Unreviewed',
        'groundTruth': 'benign' if is_benign else 'malicious',
        'similarityKey': attack_type.lower().replace(' ', '-'),
        'triggerReason': 'Flow pattern maps from a labelled CSE-CIC-IDS2018 scenario.',
        'evidence': f"Dataset label: {label}; destination port: {port}; protocol: {protocol}.",
        'recommendedAction': 'Review related traffic and confirm whether escalation is needed.',
    }

alerts = [row_to_alert(row, i + 1) for i, (_, row) in enumerate(sample_df.iterrows())]
alerts[:2]

## 8. Validate Alert Schema

In [ ]:
REQUIRED_FIELDS = [
    'id', 'timestamp', 'sourceIp', 'destinationIp', 'protocol', 'port',
    'attackType', 'severity', 'confidence', 'baseRiskScore',
    'currentRiskScore', 'status', 'groundTruth', 'similarityKey',
    'triggerReason', 'evidence', 'recommendedAction'
]
VALID_SEVERITIES = {'Critical', 'High', 'Medium', 'Low'}
VALID_GROUND_TRUTH = {'benign', 'malicious'}

def validate_alert(alert):
    missing = [field for field in REQUIRED_FIELDS if field not in alert]
    assert not missing, f"{alert.get('id')} missing fields: {missing}"
    assert alert['severity'] in VALID_SEVERITIES, f"Invalid severity: {alert['severity']}"
    assert alert['groundTruth'] in VALID_GROUND_TRUTH, f"Invalid groundTruth: {alert['groundTruth']}"
    assert alert['status'] == 'Unreviewed', f"Invalid status: {alert['status']}"
    assert 0 <= alert['confidence'] <= 100, f"Invalid confidence: {alert['confidence']}"
    assert 0 <= alert['baseRiskScore'] <= 100, f"Invalid baseRiskScore: {alert['baseRiskScore']}"
    assert 0 <= alert['currentRiskScore'] <= 100, f"Invalid currentRiskScore: {alert['currentRiskScore']}"
    assert isinstance(alert['port'], (int, float)), f"Port must be numeric: {alert['port']}"
    assert str(alert['similarityKey']).strip(), 'similarityKey must not be empty'

def validate_alerts(alerts):
    seen_ids = set()
    for alert in alerts:
        validate_alert(alert)
        assert alert['id'] not in seen_ids, f"Duplicate alert ID: {alert['id']}"
        seen_ids.add(alert['id'])
    return True

validate_alerts(alerts)
print(f'Validated {len(alerts)} alert objects.')

## 9. Export sample-alerts.json

In [ ]:
with OUTPUT_PATH.open('w', encoding='utf-8') as file:
    json.dump(alerts, file, indent=2)

print(f'Exported Colab output to: {OUTPUT_PATH}')
print('Review the file before copying it into stage-1/data/processed/sample-alerts.json.')
OUTPUT_PATH

## Preprocessing Summary

In [ ]:
print('CSV file used:', CSV_FILES_USED)
print('Rows loaded:', ROWS_LOADED)
print('Rows after cleaning:', ROWS_AFTER_CLEANING)
print('Rows sampled:', len(sample_df))
print('Benign sample count:', BENIGN_SAMPLE_COUNT)
print('Malicious sample count:', MALICIOUS_SAMPLE_COUNT)
print('Missing / invalid values removed:', ROWS_REMOVED_DURING_CLEANING)
print('Label distribution before sampling:', LABEL_DISTRIBUTION_BEFORE_SAMPLING)
print('Label distribution after sampling:', LABEL_DISTRIBUTION_AFTER_SAMPLING)
print('Output file:', OUTPUT_PATH)

## 10. Notes and Findings

Record dataset files used, sample size, label distribution, cleaning decisions, dropped or renamed columns, and the final output path here.

Initial local archive findings:

- Primary Colab source is KaggleHub dataset `solarmainframe/ids-intrusion-csv`.
- Local `archive.zip` remains supported as a fallback for offline inspection.
- `archive.zip` contains 10 CSV files.
- Most CSV files have 80 columns and do not include `Src IP` or `Dst IP`.
- `02-20-2018.csv` has 84 columns and includes `Flow ID`, `Src IP`, `Src Port`, and `Dst IP`.
- All inspected files include `Label` as the final column.